# Fetching data at warp speed

In [ ]:
# 1. Install dependencies
!pip install dukascopy-python pandas pyarrow joblib -q

import os
import time
import pandas as pd
import dukascopy_python as dp
from datetime import datetime
from pathlib import Path
from joblib import Parallel, delayed

# --- CONFIGURATION ---
# Set your output directory (Colab local disk is fast, but disappears on reset)
OUT_DIR = Path("./code/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ["AUD/USD", "NZD/USD", "EUR/NOK", "EUR/SEK", "USD/ZAR"]
MONTHS = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512",
    "202601", "202602", "202603"
]

# PARALLELISM: Use -1 to use all cores, or 4-8 to avoid getting rate-limited by Dukascopy
N_JOBS = 4 

# --- HELPER FUNCTIONS ---
def get_month_range(yyyymm: str):
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    start = datetime(year, month, 1)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)
    return start, end

def process_job(pair, yyyymm, compression="snappy"):
    """Function to be executed in parallel"""
    start, end = get_month_range(yyyymm)
    symbol = pair.replace("/", "").lower()
    
    try:
        # Ticks return both bid and ask in one fetch
        df = dp.fetch(
            instrument=pair,
            interval=dp.INTERVAL_TICK,
            offer_side=dp.OFFER_SIDE_BID, 
            start=start,
            end=end,
        )

        if df is None or len(df) == 0:
            return f"Empty: {pair} {yyyymm}"

        results_stats = []
        for side, price_col, vol_col in [("bid", "bidPrice", "bidVolume"), ("ask", "askPrice", "askVolume")]:
            out_name = f"{symbol}_dukascopy_{side}_{yyyymm}.parquet"
            out_path = OUT_DIR / out_name
            
            # Efficient DataFrame construction
            pd.DataFrame({
                "datetime": df.index,
                "symbol": symbol.upper(),
                "price_type": side.upper(),
                "price": df[price_col].values,
                "volume": df[vol_col].values,
            }).to_parquet(out_path, engine="pyarrow", compression=compression, index=False)
            
            results_stats.append(f"{out_name} ({len(df):,} rows)")
            
        return f"Success: {pair} {yyyymm} -> " + " & ".join(results_stats)
    
    except Exception as e:
        return f"FAILED: {pair} {yyyymm} | Error: {str(e)}"

# --- EXECUTION ---
jobs = [(p, m) for p in PAIRS for m in MONTHS]
print(f"🚀 Starting Parallel Download of {len(jobs)} jobs using {N_JOBS} workers...\n")

t0 = time.time()
# Parallel magic happens here
results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(process_job)(p, m) for p, m in jobs
)

# Print Summary
for r in results:
    print(r)

elapsed = time.time() - t0
print(f"\n✨ All done! Finished in {elapsed:.1f}s")
print(f"Files saved to: {OUT_DIR.absolute()}")

🚀 Starting Parallel Download of 75 jobs using 4 workers...



INFO:DUKASCRIPT:current timestamp :2025-01-02T09:13:50.981000
INFO:DUKASCRIPT:current timestamp :2025-02-03T02:20:28.814000
INFO:DUKASCRIPT:current timestamp :2025-03-03T10:20:44.109000
INFO:DUKASCRIPT:current timestamp :2025-04-01T10:42:01.509000
INFO:DUKASCRIPT:current timestamp :2025-02-03T08:25:53.211000
INFO:DUKASCRIPT:current timestamp :2025-01-02T15:53:33.384000
INFO:DUKASCRIPT:current timestamp :2025-03-03T18:49:30.680000
INFO:DUKASCRIPT:current timestamp :2025-04-01T21:05:43.677000
INFO:DUKASCRIPT:current timestamp :2025-02-03T15:00:02.150000
INFO:DUKASCRIPT:current timestamp :2025-01-03T00:31:35.337000
INFO:DUKASCRIPT:current timestamp :2025-04-02T13:02:01.654000
INFO:DUKASCRIPT:current timestamp :2025-03-04T02:23:52.069000
INFO:DUKASCRIPT:current timestamp :2025-01-03T13:11:20.742000
INFO:DUKASCRIPT:current timestamp :2025-02-03T16:56:40.870000
INFO:DUKASCRIPT:current timestamp :2025-04-02T20:43:43.980000
INFO:DUKASCRIPT:current timestamp :2025-03-04T09:19:27.802000
INFO:DUK